In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

### bar analysis_SVD

In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

DATA_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/data")

OUT_BASE = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/0410_SVD_bar_figures")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# STORAGE
# =========================

results = defaultdict(lambda: defaultdict(dict))

# =========================
# PARSE FILENAME
# =========================

def parse_filename(fname):
    if fname.startswith("ORIGINAL"):
        k = int(re.search(r'(\d+)', fname).group(1))
        return "ORIGINAL", "original", k

    match = re.match(r"f_(.*?)_(\d+u)_.*?_(\d+)recommendation\.csv", fname)
    if match:
        genre = match.group(1).replace("_s", "'s")
        n = match.group(2)
        k = int(match.group(3))
        return genre, n, k

    return None, None, None

# =========================
# METRIC
# =========================

def compute_avg(df, target_genre):
    df['has_genre'] = df['genres_all'].apply(
        lambda x: target_genre.lower() in x.lower()
    )
    user_counts = df.groupby('user_id')['has_genre'].sum()
    return user_counts.mean(), len(user_counts)

# =========================
# READ NON-ORIGINAL
# =========================

all_n_values = set()

for fname in os.listdir(DATA_DIR):
    if not fname.endswith(".csv"):
        continue

    genre, n, k = parse_filename(fname)
    if genre is None or n == "original":
        continue

    df = pd.read_csv(DATA_DIR / fname)
    avg_val, num_users = compute_avg(df, genre)

    results[genre][k][n] = (avg_val, num_users)
    all_n_values.add(n)

# =========================
# ORIGINAL FILES
# =========================

original_data = defaultdict(dict)

for fname in os.listdir(DATA_DIR):
    if not fname.startswith("ORIGINAL"):
        continue

    _, _, k = parse_filename(fname)
    df = pd.read_csv(DATA_DIR / fname)

    all_genres = set()
    df['genres_all'].str.split(',').apply(
        lambda lst: [all_genres.add(g.strip()) for g in lst]
    )

    for genre in all_genres:
        avg_val, num_users = compute_avg(df.copy(), genre)
        original_data[genre][k] = (avg_val, num_users)

# =========================
# MERGE ORIGINAL
# =========================

for genre in results.keys():
    for k in results[genre].keys():
        if genre in original_data and k in original_data[genre]:
            results[genre][k]['original'] = original_data[genre][k]

# =========================
# COLORS (FINAL STYLE PALETTE)
# =========================

def generate_colors(n_list):
    palette = [
        "#4E79A7", "#59A14F", "#E15759", "#B07AA1",
        "#F28E2B", "#76B7B2", "#EDC948", "#9C755F",
        "#FF9DA7", "#8CD17D", "#86BCB6", "#BAB0AC"
    ]
    return {n: palette[i % len(palette)] for i, n in enumerate(n_list)}

# =========================
# PLOTTING
# =========================

for genre, k_data in results.items():

    ks = sorted(k_data.keys())

    n_types = sorted(set(
        n for k in ks for n in k_data[k].keys()
    ), key=lambda x: (x != "original", int(x.replace("u", "")) if x != "original" else -1))

    color_map = generate_colors(n_types)

    x = np.arange(len(ks)) * 1.2
    width = 1.0 / len(n_types)

    fig, ax = plt.subplots(figsize=(18, 11))

    max_val = 0
    bar_containers = []

    for i, n in enumerate(n_types):
        vals = [k_data[k].get(n, (0, 0))[0] for k in ks]

        bars = ax.bar(
            x + (i - len(n_types)/2) * width + width/2,
            vals,
            width=width,
            color=color_map[n],
            edgecolor="black",
            linewidth=0.6,
            label=n
        )

        bar_containers.append((bars, vals))
        max_val = max(max_val, max(vals))

    # =========================
    # STYLE (UPDATED)
    # =========================

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    ax.set_title(f"SVD - {genre}", fontsize=40, fontweight="bold", pad=20)
    ax.set_xlabel("k (recommendation size)", fontsize=38, labelpad=14)
    ax.set_ylabel("Avg count per user", fontsize=38, labelpad=14)

    ax.set_xticks(x)
    ax.set_xticklabels(ks, fontsize=34)
    ax.tick_params(axis='y', labelsize=34)

    ax.legend(fontsize=22, ncol=3)

    # =========================
    # BAR LABELS
    # =========================

    for bars, vals in bar_containers:
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max_val * 0.015,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=22,
                rotation=90,
            )

    ax.set_ylim(top=max_val * 1.25)

    # =========================
    # SAVE FIGURE
    # =========================

    genre_folder = OUT_BASE / f"SVD_{genre.lower().replace(' ', '_')}_analysis"
    genre_folder.mkdir(parents=True, exist_ok=True)

    fig_path = genre_folder / f"SVD - {genre}.png"

    plt.tight_layout()
    plt.savefig(fig_path, dpi=300)
    plt.close()

    # =========================
    # SAVE EXPLANATION TEXT
    # =========================

    explanation_path = genre_folder / f"SVD - {genre}.txt"

    with open(explanation_path, "w") as f:
        f.write(f"SVD Analysis for Genre: {genre}\n\n")
        f.write("Metric: Average count of target genre per user\n\n")

        for k in ks:
            f.write(f"k = {k}\n")
            for n in n_types:
                val, users = k_data[k].get(n, (0, 0))
                f.write(f"  {n}: {val:.4f} (users={users})\n")
            f.write("\n")

print("All SVD figures and explanation files saved.")

All SVD figures and explanation files saved.


### param

In [2]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── Paths ──
BASE = "/home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result"
BI_DIR = os.path.join(BASE, "bi")
MU_DIR = os.path.join(BASE, "mio")
BU_DIR = os.path.join(BASE, "bu")
MERGED_DIR = os.path.join(BASE, "merged")
FIG_DIR = os.path.join(BASE, "SVD_details_figures")

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Patterns ──
BI_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_pos5_neg1_books_bi_per_genre\.csv$")

# ── Step 1: Discover bi files ──
bi_files = {}
for fname in os.listdir(BI_DIR):
    m = BI_PATTERN.match(fname)
    if m:
        genre_key = m.group(1)
        n = int(m.group(2))
        bi_files[(genre_key, n)] = fname

# ── Step 2: Merge bi + mu + bu ──
merged_data = {}

for (genre_key, n), bi_fname in sorted(bi_files.items()):
    mu_fname = bi_fname.replace("_bi_", "_mu_")
    bu_fname = bi_fname.replace("_bi_", "_bu_")

    bi_path = os.path.join(BI_DIR, bi_fname)
    mu_path = os.path.join(MU_DIR, mu_fname)
    bu_path = os.path.join(BU_DIR, bu_fname)

    if not (os.path.exists(mu_path) and os.path.exists(bu_path)):
        print(f"WARNING: Missing mu/bu for {bi_fname}, skipping.")
        continue

    df_bi = pd.read_csv(bi_path)
    df_mu = pd.read_csv(mu_path)
    df_bu = pd.read_csv(bu_path)

    merge_cols = ["genre"]
    if "is_target" in df_bi.columns:
        merge_cols.append("is_target")

    df = pd.merge(df_mu, df_bi, on=merge_cols, how="outer")
    df = pd.merge(df, df_bu, on=merge_cols, how="outer")

    merged_data[(genre_key, n)] = df

print(f"Merged {len(merged_data)} configs")

# ── Step 3: Organize by genre ──
genre_groups = defaultdict(dict)
for (genre_key, n), df in merged_data.items():
    genre_groups[genre_key][n] = df

# ── Style ──
plt.style.use("seaborn-v0_8-pastel")

TARGET_COLOR = "#5B3A29"
TARGET_LINEWIDTH = 3.5
NON_TARGET_LINEWIDTH = 1.5

PASTEL_COLORS = [
    "#AEC6CF","#FFD1DC","#B5EAD7","#FFDAC1","#C3B1E1",
    "#F3E5AB","#FFB7B2","#B0E0E6","#D5AAFF","#FFFACD",
    "#E0BBE4","#957DAD","#D4E157","#80CBC4","#F48FB1",
]

# ── Plot helper ──
def plot_figure(genre_key, sorted_ns, n_dict, target_genre_name, metric_name, compute_value, ylabel, suffix):

    # Collect all genres
    all_genres = set()
    for n in sorted_ns:
        all_genres.update(n_dict[n]["genre"].tolist())
    all_genres = sorted(all_genres)

    # Build series
    genre_series = {}
    for genre_name in all_genres:
        pts = []
        for n in sorted_ns:
            df = n_dict[n]
            row = df[df["genre"] == genre_name]
            if not row.empty:
                val = compute_value(row)
                pts.append((n, val))
        genre_series[genre_name] = pts

    non_target = [g for g in all_genres if g != target_genre_name]

    # Colors
    color_map = {}
    for i, g in enumerate(non_target):
        color_map[g] = PASTEL_COLORS[i % len(PASTEL_COLORS)]
    color_map[target_genre_name] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))
    n_to_pos = {n: p for n, p in zip(sorted_ns, positions)}

    fig, ax = plt.subplots(figsize=(14, 8))

    # Non-target
    for g in non_target:
        pts = genre_series[g]
        if pts:
            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]
            ax.plot(xs, ys, marker="o",
                    color=color_map[g],
                    linewidth=NON_TARGET_LINEWIDTH,
                    markersize=9,
                    alpha=0.8,
                    label=g)

    # Target
    pts = genre_series[target_genre_name]
    if pts:
        xs = [n_to_pos[x] for x, _ in pts]
        ys = [y for _, y in pts]
        ax.plot(xs, ys, marker="o",
                color=TARGET_COLOR,
                linewidth=TARGET_LINEWIDTH,
                markersize=7,
                zorder=10,
                label=f"{target_genre_name} (target)")

    pretty = genre_key.replace("_", " ")

    ax.set_title(f"NMF - {pretty} ({metric_name})", fontsize=26, fontweight='bold')
    ax.set_xlabel("Number of Fictitious Users (n)", fontsize=25)
    ax.set_ylabel(ylabel, fontsize=28, fontweight='bold')

    ax.set_xticks(positions)
    ax.set_xticklabels([str(n) for n in sorted_ns], rotation=45)

    ax.tick_params(axis='y', labelsize=18)
    ax.tick_params(axis='x', labelsize=22)

    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    out = os.path.join(FIG_DIR, f"{genre_key}_{suffix}.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out}")

# ── Step 4: Generate ALL figures ──
for genre_key, n_dict in sorted(genre_groups.items()):
    sorted_ns = sorted(n_dict.keys())

    sample_df = n_dict[sorted_ns[0]]
    target_rows = sample_df[sample_df["is_target"] == "target"]
    if target_rows.empty:
        print(f"WARNING: No target for {genre_key}")
        continue

    target_genre_name = target_rows["genre"].iloc[0]

    # 1. μ + bi
    plot_figure(
        genre_key, sorted_ns, n_dict, target_genre_name,
        "μ + bᵢ",
        lambda r: r["global_mean_mu"].values[0] + r["avg_bi"].values[0],
        r"$\mu + b_i$",
        "mu_plus_bi"
    )

    # 2. bi only
    plot_figure(
        genre_key, sorted_ns, n_dict, target_genre_name,
        "bᵢ",
        lambda r: r["avg_bi"].values[0],
        r"$b_i$",
        "bi_only"
    )

    # 3. bu only
    plot_figure(
        genre_key, sorted_ns, n_dict, target_genre_name,
        "bᵤ",
        lambda r: r["avg_bu"].values[0],
        r"$b_u$",
        "bu_only"
    )

    # 4. mu only
    plot_figure(
        genre_key, sorted_ns, n_dict, target_genre_name,
        "μ",
        lambda r: r["global_mean_mu"].values[0],
        r"$\mu$",
        "mu_only"
    )

print("Done!")

Merged 22 configs
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Romance_mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Romance_bi_only.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Romance_bu_only.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Romance_mu_only.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Thriller_mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_details_figures/Thriller_bi_only.png
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0323_similar_pr_details/0410/SVD-0410_lr_true/result/SVD_de